In [1]:
"""
PDF Book Chatbot with Groq LLM - Google Colab Version
======================================================
Natural ChatGPT-like responses from your PDF documents
Using Groq (Fast & Free!)

"""

# ============================================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================================

print("📦 Installing required libraries...\n")

!pip install -q PyPDF2 groq nltk scikit-learn

print("✓ Installation complete!\n")

# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import os
import re
import string
from datetime import datetime
from google.colab import files
import io
from nltk.corpus import wordnet
import PyPDF2
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from groq import Groq

# Download NLTK data
print("Downloading NLP resources...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
print("Ready!\n")

# ============================================================================
# STEP 3: CONFIGURATION - SET YOUR VALUES HERE!
# ============================================================================

# Get your FREE API key from: https://console.groq.com/
GROQ_API_KEY = "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"  # ← PASTE YOUR GROQ API KEY HERE!

# Your PDF filename (after uploading to Colab)
PDF_FILE_NAME = "/content/2023_Book_NeuralNetworksAndDeepLearning.pdf"  # ← CHANGE THIS TO YOUR PDF NAME!

# Groq model to use (fast and free!)
GROQ_MODEL = "llama-3.3-70b-versatile"  # Best balance of speed and quality

# Alternative models you can try:
# "llama-3.3-70b-versatile" - Best quality, still very fast
# "llama-3.1-70b-versatile" - Good quality, fastest
# "mixtral-8x7b-32768" - Good for long documents

# ============================================================================
# STEP 4: PDF PROCESSOR
# ============================================================================

class PDFProcessor:
    """Extract and process text from PDF files"""

    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))

    def extract_text_from_pdf(self, pdf_path):
        """Extract text from PDF file"""
        try:
            text = ""
            with open("/content/2023_Book_NeuralNetworksAndDeepLearning.pdf", 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                num_pages = len(pdf_reader.pages)

                print(f"📄 Reading: {os.path.basename(pdf_path)}")
                print(f"📖 Pages: {num_pages}")

                for page_num in range(num_pages):
                    page = pdf_reader.pages[page_num]
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"

                    if (page_num + 1) % 20 == 0:
                        print(f"   Progress: {page_num + 1}/{num_pages}")

                print(f"✓ Extracted {len(text):,} characters\n")
                return text

        except FileNotFoundError:
            print(f"PDF not found: {pdf_path}")
            print(f"   Make sure you uploaded the file to Colab!")
            return None
        except Exception as e:
            print(f"Error: {e}")
            return None

    def clean_text(self, text):
        """Clean extracted text"""
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\n\d+\n', '\n', text)
        text = re.sub(r'Page \d+', '', text)
        text = re.sub(r'\x00', '', text)
        return text.strip()

    def split_into_chunks(self, text, chunk_size=1000, overlap=200):
        """Split text into overlapping chunks"""
        sentences = sent_tokenize(text)
        chunks = []
        current_chunk = []
        current_length = 0

        for sentence in sentences:
            sentence_length = len(sentence.split())

            if current_length + sentence_length > chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))

                overlap_sentences = []
                overlap_length = 0
                for s in reversed(current_chunk):
                    if overlap_length < overlap:
                        overlap_sentences.insert(0, s)
                        overlap_length += len(s.split())
                    else:
                        break
                current_chunk = overlap_sentences
                current_length = overlap_length

            current_chunk.append(sentence)
            current_length += sentence_length

        if current_chunk:
            chunks.append(' '.join(current_chunk))

        return chunks

    def preprocess_for_search(self, text):
        """Preprocess text for searching"""
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        tokens = word_tokenize(text)
        tokens = [
            self.lemmatizer.lemmatize(token)
            for token in tokens
            if token not in self.stop_words and len(token) > 2
        ]
        return ' '.join(tokens)

# ============================================================================
# STEP 5: CHATBOT WITH GROQ LLM
# ============================================================================

class PDFChatbotGroq:
    """PDF Chatbot with Groq LLM for fast, natural responses"""
    GREETING_INPUTS = (
        "hi","hello","hey","sup","yo","hii"
    )

    GREETING_RESPONSES = [
        "Hello 👋",
        "Hey there 😊",
        "Hi! How can I help you with the book?",
    ]

    THANKS_INPUTS = ("thanks","thank you","obrigada","obrigado")
    THANKS_RESPONSES = [
        "You're welcome 😊",
        "Anytime! 🤍",
        "DE NADA !"
    ]

    def check_small_talk(self,text):
        text = text.lower()
        if text in self.GREETING_INPUTS:
          return np.random.choice(self.GREETING_RESPONSES)
        if text in self.THANKS_INPUTS:
          return np.random.choice(self.THANKS_RESPONSES)
        for w in text.split():
            if w in self.GREETING_INPUTS:
                return np.random.choice(self.GREETING_RESPONSES)
        for w in text.split():
            if w in self.THANKS_INPUTS:
                return np.random.choice(self.THANKS_RESPONSES)
        return None
    def translate_to_english(self, text):
        """Detect & translate Arabic / Portuguese → English using Groq"""
        try:
            chat_completion = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a translation assistant. Detect the language and translate the text to clear English ONLY. Do not explain. Just return translated text."
                    },
                    {
                        "role": "user",
                        "content": text
                    }
                ],
                temperature=0
            )
            return chat_completion.choices[0].message.content.strip()
        except:
            return text
    def expand_query(self, query):
        """Query Expansion using synonyms + variations"""
        from nltk.corpus import wordnet

        q = query.lower()

        expanded_terms = set()
        words = q.split()

        # Add original words
        for w in words:
            expanded_terms.add(w)

            # Lemmatized form
            expanded_terms.add(self.processor.lemmatizer.lemmatize(w))

            # WordNet synonyms
            for syn in wordnet.synsets(w):
                for lemma in syn.lemmas():
                    term = lemma.name().replace("_", " ").lower()
                    if len(term) > 2:
                        expanded_terms.add(term)

        # Domain–specific boosting
        NN_MAP = {
            "nn": ["neural network","neural networks","ann","artificial neural network"],
            "ann": ["neural network","neural networks"],
            "deep learning": ["neural networks","machine learning","deep neural networks"],
            "ml": ["machine learning","learning systems"],
            "ai": ["artificial intelligence","machine intelligence"]
        }

        for key, values in NN_MAP.items():
            if key in q:
                for v in values:
                    expanded_terms.add(v)

        expanded_query = " ".join(list(expanded_terms))
        return expanded_query
    def __init__(self, api_key, model=GROQ_MODEL):
        self.processor = PDFProcessor()
        self.document_text = None
        self.chunks = []
        self.processed_chunks = []
        self.vectorizer = None
        self.tfidf_matrix = None
        self.is_loaded = False
        self.document_name = ""
        self.model = model

        # Initialize Groq
        try:
            self.client = Groq(api_key=api_key)
            self.use_llm = True
            print(f"Groq LLM enabled ({model})")
            print("ChatGPT-like responses activated!\n")
        except Exception as e:
            print(f"Groq Error: {e}")
            self.use_llm = False

    def load_document(self, file_path):
        """Load and process PDF"""
        print("=" * 70)
        print("LOADING DOCUMENT")
        print("=" * 70 + "\n")

        self.document_text = self.processor.extract_text_from_pdf(file_path)

        if not self.document_text:
            return False

        self.document_name = os.path.basename(file_path)

        print("Cleaning text...")
        self.document_text = self.processor.clean_text(self.document_text)

        print("Creating searchable chunks...")
        self.chunks = self.processor.split_into_chunks(self.document_text)
        print(f"Created {len(self.chunks)} chunks\n")

        print("Building search index...")
        self.processed_chunks = [
            self.processor.preprocess_for_search(chunk)
            for chunk in self.chunks
        ]

        print("Creating TF-IDF vectors...")
        self.vectorizer = TfidfVectorizer(
            max_features=1000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.8
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.processed_chunks)

        self.is_loaded = True

        print("\n" + "=" * 70)
        print("DOCUMENT READY!")
        print("=" * 70)
        print(f"Document: {self.document_name}")
        print(f"Chunks: {len(self.chunks)}")
        print(f"Words: ~{len(self.document_text.split()):,}")
        print(f"Mode: Natural responses (Groq LLM)")
        print(f"Model: {self.model}")
        print("=" * 70 + "\n")

        return True

    def find_relevant_chunks(self, query, top_k=5):
        """Find relevant chunks using TF-IDF similarity + Query Expansion"""
        if not self.is_loaded:
           return []

        #  Expand query (WordNet + domain)
        expanded = self.expand_query(query)

        # Preprocess normally
        processed_query = self.processor.preprocess_for_search(expanded)

        # Vectorize
        query_vector = self.vectorizer.transform([processed_query])
        similarities = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        results = []
        for idx in top_indices:
            if similarities[idx] > 0.05:
               results.append({
                  'chunk': self.chunks[idx],
                  'score': similarities[idx],
                  'index': idx
              })
        return results

    def generate_response(self, query):
        """Generate natural response using Groq"""
        talk = self.check_small_talk(query)
        if talk:
            return talk
        # Find relevant chunks
        translated_query = self.translate_to_english(query)
        relevant_chunks = self.find_relevant_chunks(query, top_k=5)

        if not relevant_chunks:
            return "I couldn't find relevant information in the document to answer your question. Try asking about topics covered in the text."

        # Combine context from top chunks
        context = "\n\n".join([chunk['chunk'] for chunk in relevant_chunks[:3]])

        # Build prompt for Groq
        system_prompt = """You are a helpful AI assistant answering questions about a document.

Instructions:
- Answer naturally and conversationally, like ChatGPT
- Base your answer on the context provided below
- Be clear, concise, and informative (2-4 paragraphs)
- Use examples when helpful
- If the context doesn't fully answer, explain what you can answer
- Don't say "according to the document" or "the text says" - just answer naturally
- Write in a friendly, educational tone"""

        user_prompt = f"""Context from the document:

{context}

Question: {query}

Please answer the question based on the context above."""

        try:
            # Call Groq API
            chat_completion = self.client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                model=self.model,
                temperature=0.7,
                max_tokens=1500,
                top_p=1,
                stream=False
            )

            answer = chat_completion.choices[0].message.content

            # Add source reference
            confidence = relevant_chunks[0]['score']
            if confidence > 0.4:
                confidence_emoji = "✅"
            elif confidence > 0.2:
                confidence_emoji = "⚠️"
            else:
                confidence_emoji = "⚡"

            answer += f"\n\n{confidence_emoji} *Based on sections {relevant_chunks[0]['index']+1}"
            if len(relevant_chunks) > 1:
                answer += f"-{relevant_chunks[-1]['index']+1}"
            answer += f" of {len(self.chunks)}*"

            return answer

        except Exception as e:
            return f"Error: {e}\n\nShowing relevant text:\n\n{relevant_chunks[0]['chunk'][:500]}..."

    def ask(self, query):
        """Ask a question"""
        if not self.is_loaded:
            print("No document loaded!")
            return

        print(f"\n{'='*70}")
        print(f"You: {query}")
        print(f"{'='*70}\n")
        print("FAABot: [Thinking...]\n")

        response = self.generate_response(query)

        print(f"FAABot:\n\n{response}\n")
        print("=" * 70 + "\n")

        return response

    def chat(self):
        """Interactive chat mode"""
        if not self.is_loaded:
            print("Load a document first!")
            return

        print("\n" + "=" * 70)
        print("         INTERACTIVE CHAT MODE (Groq LLM)")
        print("=" * 70)
        print("\nAsk questions naturally - type 'exit' to quit")
        print("⚡ Powered by Groq - Lightning fast responses!")
        print("=" * 70 + "\n")

        while True:
            try:
                user_input = input("You: ").strip()

                if not user_input:
                    continue

                if user_input.lower() in ['exit', 'quit', 'bye', 'stop']:
                    print("\nFAABot: Goodbye! Happy learning! 👋\n")
                    break

                print("\nFAABot: [Thinking...]\n")
                response = self.generate_response(user_input)
                print(f"FAABot:\n\n{response}\n")
                print("-" * 70 + "\n")

            except KeyboardInterrupt:
                print("\n\nFAABot: Goodbye!")
                break

    def summary(self):
        """Get document summary"""
        if not self.is_loaded:
            print("No document loaded")
            return

        sentences = sent_tokenize(self.document_text)[:5]
        preview = ' '.join(sentences)

        print(f"""
DOCUMENT INFORMATION
{'='*70}
Name: {self.document_name}
Chunks: {len(self.chunks)}
Words: ~{len(self.document_text.split()):,}
Mode: Natural ChatGPT-like responses
LLM: Groq ({self.model})

📖 PREVIEW:
{preview}...
{'='*70}
""")

# ============================================================================
# STEP 6: HELPER FUNCTIONS FOR COLAB
# ============================================================================

def upload_pdf():
    """Upload PDF file to Colab"""
    print("Please select your PDF file to upload:\n")
    uploaded = files.upload()

    if uploaded:
        filename = list(uploaded.keys())[0]
        print(f"\nUploaded: {filename}")
        return filename
    else:
        print("\n No file uploaded")
        return None

def create_bot(pdf_filename=None, api_key=None):
    """Create chatbot - easy one-step function"""

    # Check API key
    if api_key is None:
        api_key = GROQ_API_KEY

    if api_key == "your-api-key-here":
        print("=" * 70)
        print("NO API KEY FOUND!")
        print("=" * 70)
        print("\nTo get ChatGPT-like responses with Groq (FREE!):")
        print("1. Go to: https://console.groq.com/")
        print("2. Sign up (FREE - no credit card needed)")
        print("3. Create an API key")
        print("4. Paste it in GROQ_API_KEY variable above")
        print("\n💡 Groq is FAST and has generous free tier!")
        print("\nFor now, you can:")
        api_key = input("Enter Groq API key now (or press Enter to skip): ").strip()
        if not api_key:
            print("\nCannot proceed without API key")
            return None

    # Create bot
    bot = PDFChatbotGroq(api_key=api_key)

    # Check PDF
    if pdf_filename is None:
        pdf_filename = PDF_FILE_NAME

    if pdf_filename == "your_book.pdf" or not os.path.exists(pdf_filename):
        print("\nPDF not found. Let's upload one!")
        uploaded_file = upload_pdf()
        if uploaded_file:
            pdf_filename = uploaded_file
        else:
            return None

    # Load document
    success = bot.load_document(pdf_filename)

    if success:
        print("Chatbot ready! Try these commands:")
        print("   bot.ask('your question here')")
        print("   bot.chat()  # Interactive mode")
        print("   bot.summary()  # Document info\n")
        return bot
    else:
        print("Failed to create chatbot")
        return None

# ============================================================================
# STEP 7: WELCOME MESSAGE
# ============================================================================

print("\n" + "=" * 70)
print("         PDF CHATBOT WITH GROQ LLM - COLAB VERSION")
print("=" * 70)
print("""
⚡ Why Groq?
- FREE tier with generous limits
- FAST responses (10x faster than others!)
- No credit card needed
- High-quality answers

Quick Start Guide:

1. Get FREE API key:
   👉 https://console.groq.com/
   (Sign up takes 30 seconds, no credit card!)

2. Set your API key above:
   GROQ_API_KEY = "gsk_..."

3. Create bot:
   bot = create_bot()

4. Ask questions naturally:
   bot.ask('What is machine learning?')
   bot.ask('Explain deep learning in simple terms')

5. Interactive chat:
   bot.chat()

💡 Get natural, ChatGPT-like answers from your PDF!
⚡ Powered by Groq - Lightning fast and FREE!
""")
print("=" * 70 + "\n")

# Uncomment the line below to auto-start:
# bot = create_bot()

📦 Installing required libraries...

✓ Installation complete!

Ready!


         PDF CHATBOT WITH GROQ LLM - COLAB VERSION

⚡ Why Groq?
- FREE tier with generous limits
- FAST responses (10x faster than others!)
- No credit card needed
- High-quality answers

Quick Start Guide:

1. Get FREE API key:
   👉 https://console.groq.com/
   (Sign up takes 30 seconds, no credit card!)

2. Set your API key above:
   GROQ_API_KEY = "gsk_..."

3. Create bot:
   bot = create_bot()

4. Ask questions naturally:
   bot.ask('What is machine learning?')
   bot.ask('Explain deep learning in simple terms')

5. Interactive chat:
   bot.chat()

💡 Get natural, ChatGPT-like answers from your PDF!
⚡ Powered by Groq - Lightning fast and FREE!




In [2]:
bot = create_bot()


✓ Groq LLM enabled (llama-3.3-70b-versatile)🚀
  ChatGPT-like responses activated!

LOADING DOCUMENT

📄 Reading: 2023_Book_NeuralNetworksAndDeepLearning.pdf
📖 Pages: 512
   Progress: 20/512
   Progress: 40/512
   Progress: 60/512
   Progress: 80/512
   Progress: 100/512
   Progress: 120/512
   Progress: 140/512
   Progress: 160/512
   Progress: 180/512
   Progress: 200/512
   Progress: 220/512
   Progress: 240/512
   Progress: 260/512
   Progress: 280/512
   Progress: 300/512
   Progress: 320/512
   Progress: 340/512
   Progress: 360/512
   Progress: 380/512
   Progress: 400/512
   Progress: 420/512
   Progress: 440/512
   Progress: 460/512
   Progress: 480/512
   Progress: 500/512
✓ Extracted 1,580,730 characters

Cleaning text...
Creating searchable chunks...
Created 311 chunks

Building search index...
Creating TF-IDF vectors...

DOCUMENT READY!
Document: 2023_Book_NeuralNetworksAndDeepLearning.pdf
Chunks: 311
Words: ~240,110
Mode: Natural responses (Groq LLM)
Model: llama-3.3-70b-ve

In [ ]:
bot.chat()


         INTERACTIVE CHAT MODE (Groq LLM)

Ask questions naturally - type 'exit' to quit
⚡ Powered by Groq - Lightning fast responses!


FAABot: [Thinking...]

FAABot:

Visual attention refers to the ability of a system to focus on specific parts of an image or a scene, rather than processing the entire image at once. This is inspired by the way the human eye works, where the retina has a high-resolution central region called the fovea, which is responsible for detailed vision, and a lower-resolution peripheral region. In the context of computer vision, visual attention mechanisms are used to selectively focus on certain regions of an image, allowing the system to process and understand the most relevant information.

In essence, visual attention is a way to mimic the human visual system's ability to selectively focus on certain parts of the visual field, rather than processing the entire scene at once. This can be particularly useful in applications such as image recognition, object 